In [1]:
%load_ext rpy2.ipython

In [2]:
import pandas as pd

import src
import src.load

r_colormap = src.r_colormap

In [3]:
%%R -i r_colormap

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /Users/lukas/git/ytpop


In [4]:
pd.options.display.float_format = "{:.1f}".format

In [5]:
channel_df = src.load.channels()
video_df = src.load.videos(filter_period=True, filter_sentences=False)
sentence_df = src.load.sentences(filter_video=False)

sents = sentence_df.groupby("video_id", observed=True).agg(n_sents=("video_id", "size"))

df = pd.merge(
    channel_df,
    video_df,
    on="channel_id",
    how="inner",
)

df = pd.merge(df, sents, on="video_id")

# per Channel (all videos)

In [8]:
channel_overview = df.groupby("channel", observed=True).agg(
    videos=("channel", "size"),
    follower_count=("channel_follower_count", "first"),
    n_sentences=("n_sents", "sum"),
    first_video=("datetime_upload", "min"),
    latest_video=("datetime_upload", "max"),
)

In [10]:
channel_overview

,videos,follower_count,n_sentences,first_video,latest_video
channel,,,,,
AfD TV,1454,250000,142870,2017-12-07,2024-01-19
AfD BT,5220,388000,295280,2017-12-06,2024-01-20
Greens,457,26100,45677,2018-01-27,2023-12-13
CDU,632,21900,53112,2017-12-11,2024-01-19
CSU,145,5170,10357,2017-12-14,2023-10-05
Left,435,29000,45942,2017-12-11,2024-01-17
FDP,483,23300,36204,2018-01-06,2024-01-06
SPD,477,24200,65563,2017-12-07,2024-01-18


In [5]:
query_all = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.format == "videos",
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count.label("followers"),
        func.count(Video.id.distinct()).label("videos"),
        func.count(Sentence.id).label("sentences"),
        func.min(Video.datetime_upload).label("first_video"),
        func.max(Video.datetime_upload).label("latest_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)



with engine.connect() as conn:
    df_all = pd.read_sql(query_all.statement, conn)

df_all.channel = df_all.channel.replace(party_names)
df_all

,channel,followers,videos,sentences,first_video,latest_video
0,AfD BT,388000,5257,318328,2017-12-06,2024-01-26
1,AfD TV,250000,1563,160132,2017-06-13,2024-01-26
2,Left,29000,1499,157693,2008-12-18,2024-01-22
3,Greens,26100,1623,119187,2008-05-07,2024-01-27
4,SPD,24200,1625,191038,2008-05-08,2024-01-24
5,FDP,23300,1952,159493,2007-09-06,2024-01-06
6,CDU,21900,2111,142666,2008-08-22,2024-01-27
7,CSU,5170,730,42547,2008-09-09,2023-10-05


# per Channel (clean)

In [6]:
query_after = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.is_valid == True,
        Sentence.is_valid == True,
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count.label("ch_followers"),
        func.count(Video.id.distinct()).label("ch_videos"),
        func.count(Sentence.id).label("n_sentences"),
        func.sum(Sentence.elite.cast(Integer)).label("n_sent_elite"),
        func.sum(Sentence.pplcentr.cast(Integer)).label("n_sent_pplcentr"),
        # func.min(Video.datetime_upload).label("first_video"),
        # func.max(Video.datetime_upload).label("latest_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)
with engine.connect() as conn:
    df_valid = pd.read_sql(query_after.statement, conn)

df_valid.channel = df_valid.channel.replace(party_names)

df_valid

,channel,ch_followers,ch_videos,n_sentences,n_sent_elite,n_sent_pplcentr
0,AfD BT,388000,5207,299414,40413,6117
1,AfD TV,250000,1457,145525,17477,3663
2,Left,29000,435,46823,2499,1425
3,Greens,26100,459,46964,1463,1358
4,SPD,24200,480,66999,1433,2253
5,FDP,23300,487,36930,1400,929
6,CDU,21900,629,54267,947,1450
7,CSU,5170,145,10463,290,239


In [11]:
# sum of durations

sum_of_seconds = df.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 1482.92 hours


In [13]:
# number of videos

count_videos = df.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 9303


In [14]:
# number of sentencs

count_sents = df.n_sents.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 695005


# per Video

In [10]:
comments_query = (
    Query(Comment.id, func.count(Comment.id))
    .filter(Comment.video_id == Video.id, Comment.is_valid == True)
    .with_entities(func.count(Comment.id))
    .scalar_subquery()
)
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True)
    .with_entities(
        Channel.channel,
        Video.datetime_upload,
        Video.like_count.label("likes"),
        Video.view_count.label("views"),
        Video.duration.label("duration"),
        comments_query.label("comments"),
    )
)

with engine.connect() as conn:
    df = pd.read_sql(query.statement, conn)
df.channel = df.channel.replace(party_names)

In [11]:
df_videos = df.groupby("channel").mean(numeric_only=True).stack().reset_index()

pivot_videos = pd.pivot(df_videos, index="channel", columns="level_1", values=0)
pivot_videos.columns = [f"avg_{col.lstrip('@')}" for col in pivot_videos.columns]
pivot_videos = pivot_videos.reset_index()

In [12]:
pivot_videos

,channel,avg_comments,avg_duration,avg_likes,avg_views
0,AfD BT,398.3,436.2,3867.4,45832.0
1,AfD TV,398.7,657.2,3627.1,43363.7
2,CDU,41.2,616.0,64.4,9624.1
3,CSU,7.5,442.2,35.6,22332.8
4,FDP,0.7,620.3,0.5,5874.0
5,Greens,0.2,892.2,79.7,4537.7
6,Left,45.0,832.4,255.8,10426.9
7,SPD,25.0,1088.2,102.1,5065.4


In [13]:
df_valid

,channel,ch_followers,ch_videos,n_sentences,n_sent_elite,n_sent_pplcentr
0,AfD BT,388000,5207,299414,40413,6117
1,AfD TV,250000,1457,145525,17477,3663
2,Left,29000,435,46823,2499,1425
3,Greens,26100,459,46964,1463,1358
4,SPD,24200,480,66999,1433,2253
5,FDP,23300,487,36930,1400,929
6,CDU,21900,629,54267,947,1450
7,CSU,5170,145,10463,290,239


In [33]:
summary_table = pd.merge(df_valid, pivot_videos, on="channel").T
summary_table.columns = [col.lstrip("@") for col in summary_table.iloc[0,:]]
summary_table = summary_table.iloc[1:,:]
summary_table = summary_table

In [34]:
summary_table

,AfD BT,AfD TV,Left,Greens,SPD,FDP,CDU,CSU
ch_followers,388000,250000,29000,26100,24200,23300,21900,5170
ch_videos,5207,1457,435,459,480,487,629,145
n_sentences,299414,145525,46823,46964,66999,36930,54267,10463
n_sent_elite,40413,17477,2499,1463,1433,1400,947,290
n_sent_pplcentr,6117,3663,1425,1358,2253,929,1450,239
avg_comments,398.3,398.7,45.0,0.2,25.0,0.7,41.2,7.5
avg_duration,436.2,657.2,832.4,892.2,1088.2,620.3,616.0,442.2
avg_likes,3867.4,3627.1,255.8,79.7,102.1,0.5,64.4,35.6
avg_views,45832.0,43363.7,10426.9,4537.7,5065.4,5874.0,9624.1,22332.8


In [14]:
path = src.PATH / "overleaf/tables/summary.tex"

summary_table.to_latex(
    path,
    float_format= "{:.1f}".format,
    escape=True,
    column_format="lrrrrrrrr",
)

summary_table.to_csv(path.with_suffix(".csv"), index=True)

NameError: name 'summary_table' is not defined

In [17]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = likes + 1,
      views = views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=2)

ggsave(here("overleaf/img/view_count_violin.pdf"))
ggsave(here("overleaf/img/view_count_violin.svg"))

Saving 13.9 x 8.33 in image
Saving 13.9 x 8.33 in image


In [18]:
%%R -i df -i r_colormap -w 800 -h 1000

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)
df_plot <- df %>%
   mutate(
      likes = likes + 1,
      views = views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=1)

ggsave(here("overleaf/img/view_count_violin_vertical.pdf"))
ggsave(here("overleaf/img/view_count_violin_vertical.svg"))

Saving 11.1 x 13.9 in image
Saving 11.1 x 13.9 in image
